In [1]:
import numpy as np
import pyomo.environ as pyo
import matplotlib.pyplot as plt

In [6]:
grid = np.zeros((3,3))
grid[0,1] = 1
grid[1,1] = 1
grid[2,1] = 1
grid[1,2] = 1
print(grid)

origem = (0, 0)
destino = (2, 2)


[[0. 1. 0.]
 [0. 1. 1.]
 [0. 1. 0.]]


In [ ]:
vizinhos = {}
for i in range(grid.shape[0]):
    for j in range(grid.shape[1]):
        vizinho_viaveis = []
        # print(f"Vizinhos de ({i}, {j}):")
        cima = (i-1, j)
        baixo = (i+1, j)
        esquerda = (i, j-1)
        direita = (i, j+1)
        vizinho= [cima, baixo, esquerda, direita]
        for v in vizinho:
            # print(v[0], v[1])
            if v[0] >=0 and v[0] < grid.shape[0] and v[1] >= 0 and v[1] < grid.shape[1]:
                vizinho_viaveis.append(v)               
        vizinhos[(i, j)] = vizinho_viaveis

for k in vizinhos:
    print(k, vizinhos[k])

(0, 0) [(1, 0), (0, 1)]
(0, 1) [(1, 1), (0, 0), (0, 2)]
(0, 2) [(1, 2), (0, 1)]
(1, 0) [(0, 0), (2, 0), (1, 1)]
(1, 1) [(0, 1), (2, 1), (1, 0), (1, 2)]
(1, 2) [(0, 2), (2, 2), (1, 1)]
(2, 0) [(1, 0), (2, 1)]
(2, 1) [(1, 1), (2, 0), (2, 2)]
(2, 2) [(1, 2), (2, 1)]


In [162]:
# criar os arcos do grafo
custo_passo = 1
arcos = []
arcos_custo = {}
keys = list(vizinhos.keys())
for i,(j,k) in enumerate(vizinhos.items()):
    # print(i, j, k)
    for jk in k:
        # print(f"Arco: {j} -> {jk}")
        
        soma = grid[jk[0], jk[1]] + grid[j[0], j[1]] + custo_passo
        arcos_custo[(j, jk)] = soma
        arcos.append((j, jk))

In [154]:
grid

array([[0., 1., 0.],
       [0., 1., 1.],
       [0., 1., 0.]])

In [166]:
arcos_custo[(0,0),(0,1)]

np.float64(2.0)

In [155]:
arcos

[((0, 0), (1, 0)),
 ((0, 0), (0, 1)),
 ((0, 1), (1, 1)),
 ((0, 1), (0, 0)),
 ((0, 1), (0, 2)),
 ((0, 2), (1, 2)),
 ((0, 2), (0, 1)),
 ((1, 0), (0, 0)),
 ((1, 0), (2, 0)),
 ((1, 0), (1, 1)),
 ((1, 1), (0, 1)),
 ((1, 1), (2, 1)),
 ((1, 1), (1, 0)),
 ((1, 1), (1, 2)),
 ((1, 2), (0, 2)),
 ((1, 2), (2, 2)),
 ((1, 2), (1, 1)),
 ((2, 0), (1, 0)),
 ((2, 0), (2, 1)),
 ((2, 1), (1, 1)),
 ((2, 1), (2, 0)),
 ((2, 1), (2, 2)),
 ((2, 2), (1, 2)),
 ((2, 2), (2, 1))]

In [122]:
matriz_soma = {}
for k in keys:
    matriz_soma[k] = 0

In [177]:
matriz_soma[(0,0)]= -1
matriz_soma[(2,2)]= 1
matriz_soma

{(0, 0): -1,
 (0, 1): 0,
 (0, 2): 0,
 (1, 0): 0,
 (1, 1): 0,
 (1, 2): 0,
 (2, 0): 0,
 (2, 1): 0,
 (2, 2): 1}

In [181]:
sum(matriz_soma.values())

0

In [183]:
model = pyo.ConcreteModel()

model.k = pyo.Set(initialize=keys)
model.arcos_custo = pyo.Param(arcos, initialize=arcos_custo)
model.matriz_s = pyo.Param(model.k, initialize=matriz_soma)
# model.arcos = pyo.Param(initialize=arcos)
# model.x = pyo.Var(model.arcos)

In [175]:
model.pprint()

1 Set Declarations
    k : Size=1, Index=None, Ordered=Insertion
        Key  : Dimen : Domain : Size : Members
        None :     2 :    Any :    9 : {(0, 0), (0, 1), (0, 2), (1, 0), (1, 1), (1, 2), (2, 0), (2, 1), (2, 2)}

2 Param Declarations
    arcos_custo : Size=24, Index={(0, 0, 1, 0), (0, 0, 0, 1), (0, 1, 1, 1), (0, 1, 0, 0), (0, 1, 0, 2), (0, 2, 1, 2), (0, 2, 0, 1), (1, 0, 0, 0), (1, 0, 2, 0), (1, 0, 1, 1), (1, 1, 0, 1), (1, 1, 2, 1), (1, 1, 1, 0), (1, 1, 1, 2), (1, 2, 0, 2), (1, 2, 2, 2), (1, 2, 1, 1), (2, 0, 1, 0), (2, 0, 2, 1), (2, 1, 1, 1), (2, 1, 2, 0), (2, 1, 2, 2), (2, 2, 1, 2), (2, 2, 2, 1)}, Domain=Any, Default=None, Mutable=False
        Key          : Value
        (0, 0, 0, 1) :   2.0
        (0, 0, 1, 0) :   1.0
        (0, 1, 0, 0) :   2.0
        (0, 1, 0, 2) :   2.0
        (0, 1, 1, 1) :   3.0
        (0, 2, 0, 1) :   2.0
        (0, 2, 1, 2) :   2.0
        (1, 0, 0, 0) :   1.0
        (1, 0, 1, 1) :   2.0
        (1, 0, 2, 0) :   1.0
        (1, 1, 0, 1) :  